<a href="https://colab.research.google.com/github/evkoff/DI-Bootcamp-Stage1/blob/main/Week12/Day1/Tutorial/W12D1_Tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GenAI_197 - W12D1 - Tutorial

## 1. Choosing our Model & Dataset

In [ ]:
MY_MODEL="openai-community/gpt2"
MY_DATASET="mteb/tweet_sentiment_extraction"
SEED=42

## 2. Data Loading and Exploration

In [ ]:
from datasets import load_dataset
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
# import evaluate
import numpy as np

dataset = load_dataset(MY_DATASET)

# Load only a subset of the dataset (so the training phase doesnt take too long)
train_data = dataset["train"].shuffle(seed=SEED).select(range(500))
test_data = dataset["test"].shuffle(seed=SEED).select(range(200))

README.md:   0%|          | 0.00/6.83k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.86M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/240k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/26732 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3432 [00:00<?, ? examples/s]

In [ ]:
train_data[6]

{'id': 'f3a526931d',
 'text': 'Saying goodbye to a good trusted friend today. Goodbye free Sky TV, you were the best friend anyone could have had.',
 'label': 2,
 'label_text': 'positive'}

## 3. Data Preprocessing: Tokenization

In [ ]:
model_name = MY_MODEL
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [ ]:
tokenizer.pad_token = tokenizer.eos_token
def tokenize_batch(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=64)

train_data = train_data.map(tokenize_batch, batched=True)
test_data = test_data.map(tokenize_batch, batched=True)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

## 4. Model Initialization

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MY_MODEL, num_labels=3)

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

KeyboardInterrupt: 

## 5. Evaluation

In [ ]:
!pip install evaluate -q

In [ ]:
import evaluate

# Evaluation metric
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=predictions, references=labels)

## 6. Fine-Tuning

In [ ]:
base_model = model

In [ ]:
# Training arguments optimized for CPU
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    fp16=torch.cuda.is_available(),
    report_to="none",
    save_total_limit=1
)

# Trainer setup
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
    compute_metrics=compute_metrics,
)

# Train the model
trainer.train()